# Exp3 Data Analysis

author: Tingying He, date: 20230325


This script is used for data preprocessing in Experiment 3 for the our paper "Design Characterization for Black-and-White Textures in Visualization."

To protect participants' personal information, please run another Python script "Prolific ID encryption" before executing this script. It will handle the encryption of Prolific IDs by replacing Prolific IDs to unique participant ids.

## Input data:
All the following data are stored in the folder final_data.
1. Exp3 results download from our server (.csv)
    * measurements.csv
        * education
        * readability score
        * BeauVis score
    * individual_texture: measurements of trials per participant (participant_id.csv), each trial is one row
        * accuracy
        * response time
    
2. Exp3 demographic data exported from Prolific (prolific_export_641f4d6c66bf43f77dc91ce4.csv)
    * gender
    * age

## Output
All generated .csv are in the generated_csv folder. Specifically, the files are used as input for CI analysis will also be written into the CI-analysis/exp-data folder. We used R to calculate CIs. The R scripts are located in the CI-analysis folder. 

### Basic information
1. Valid responses of Exp3 (a .csv file without Prolific ID)
2. Number of participants in each condition (bar/pie)
3. Demographics (gender, age, highest education)

### Files for CI analysis
4. Seperated data files (correct_rate.csv, response_time.csv, beauvis.csv, readable.csv). These files are used as input for CI analysis. We used R to calculate CIs. The R scripts are located in the CI-analysis folder. These four files will also be written into the CI-analysis/exp-data folder.


For correct rate, we averaged correct rate for all trials for each participant in each condition. 

For response time, we only counted the correct trials of participants who have reached a 90% overall correct rate (correct rate >= 90%)

For BeauVis score readability, we only counted the trials of participants who have reached a 90% overall correct rate (correct rate >= 90%)

We explain this analysis in detail in our paper and the appendix.
    

In [76]:
# import lib
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as st
import math
import os
import glob # read csv files

# Basic information

## Import data

In [77]:
# import data from LimeSurvey
df=pd.read_csv("final_data/measurements.csv")
df.head()

,participant_id,study_id,session_id,condition,is_debug,exclude_reloaders,timestamp_0,browser_name,browser_version,os,...,timestamp_98,education,gender,ethnicity,age,fluency,attention_check,commitment_ver,distraction,optionalComments
0,ialmjuh,test_study_id,test_session_id,pie,False,1,1763173291530,Chrome,142.0.0.0,macOS,...,1763173894415,none,male,white,18to24,native,Broccoli,all,attention,NaN
1,ohqrfkd,test_study_id,test_session_id,pie,False,1,1763176592422,Chrome,142.0.0.0,Windows,...,1763177112979,bachelor,female,white,18to24,fluent,Broccoli,all,attention,NaN
2,agosyvi,test_study_id,test_session_id,bar,False,1,1763189362051,Chrome,140.0.0.0,macOS,...,1763190383396,bachelor,male,hispanic,25to34,fluent,Broccoli,all,attention,crashing page after completion on the first time
3,fkzvuch,test_study_id,test_session_id,pie,False,1,1763353832903,Firefox,145.0,Windows,...,1763354718378,bachelor,male,asian,18to24,fluent,Broccoli,all,attention,NaN
4,jqulntz,test_study_id,test_session_id,pie,False,1,1763400157429,Safari,17.6,macOS,...,1763400883279,Currently in college,male,white,18to24,native,Broccoli,all,attention,NaN


In [78]:
# # import data from Prolific
# df_p=pd.read_csv("final_data/prolific_export_641f4d6c66bf43f77dc91ce4.csv")
# df_p.head()

In [79]:
# Set the folder path (for importing the files in the individual_texture folder)
folder_path = 'final_data/individual_texture'

In [80]:
# add demographics data from Prolific(df_p) to df by prolific id

# # columns we want to get from Prolific
# df['gender'] = ''
# df['age'] = ''

# # write these columns
# for index, row in df.iterrows():
#     for index_p, row_p in df_p.iterrows():
#         if row['participant_id'] == row_p['Participant id']:
#             df.loc[index, 'gender'] = row_p['Sex']
#             df.loc[index, 'age'] = row_p['Age']

In [81]:
# check condition balance
df['condition'].value_counts()

condition
bar    18
pie    14
Name: count, dtype: int64

## Select valid data

In [82]:
df.to_csv('generated_csv/exp3_valid_responses.csv', index=False)

In [83]:
# drop the 'Prolific Id' column
df_no_PID = df.drop('participant_id', axis=1)
df_no_PID.to_csv('generated_csv/exp3_valid_responses_without_prolificID.csv', index=False)

## Demographics

In [84]:
def printDemographics(df):
    #gender
    male_count = len(df[df['gender'] == 'male'])
    female_count = len(df[df['gender'] == 'female'])
    other_count = len(df[(df['gender'] != 'male') & (df['gender'] != 'female')])
    print("Gender")
    print("male:", male_count)
    print("female:", female_count)
    print("other(please check):", other_count)
    print()
    
    # age
    print("Age")
    eightteentotewentyfour_count = len(df[df['age'] == '18to24'])
    twentyfourto34_count = len(df[df['age'] == '25to34'])
    over35_count = len(df[df['age'] == '35above'])
    print("18 to 24:", eightteentotewentyfour_count)
    print("25 to 34:", twentyfourto34_count)
    print("over 35:", over35_count)
    # print("mean:", np.mean(df['age']))
    # print("sd:", df['age'].std())
    print()
    

    bachelor_count = len(df[df['education'] == 'bachelor'])
    master_count = len(df[df['education'] == 'master'])
    phd_count = len(df[df['education'] == 'phd'])
    other_count = len(df[(df['education'] != 'bachelor') & (df['education'] != 'master') & (df['education'] != 'phd')])

    print("Eductaion")
    print("bachelor:", bachelor_count)
    print("master:", master_count)
    print("phd:", phd_count)
    print("other:", other_count)
    print()

In [85]:
printDemographics(df)

Gender
male: 19
female: 11
other(please check): 2

Age
18 to 24: 26
25 to 34: 4
over 35: 2

Eductaion
bachelor: 17
master: 2
phd: 0
other: 13



In [86]:
printDemographics(df[df['condition'] == 'bar'])

Gender
male: 11
female: 6
other(please check): 1

Age
18 to 24: 14
25 to 34: 2
over 35: 2

Eductaion
bachelor: 9
master: 2
phd: 0
other: 7



In [87]:
len(df[df['condition'] == 'bar'])

18

In [88]:
len(df[df['condition'] == 'pie'])

14

In [89]:
printDemographics(df[df['condition'] == 'pie'])

Gender
male: 8
female: 5
other(please check): 1

Age
18 to 24: 12
25 to 34: 2
over 35: 0

Eductaion
bachelor: 8
master: 0
phd: 0
other: 6



# Data analysis

### Accuracy (correct rate)

In [90]:
# Find all CSV files in the individual_texture folder
csv_files = glob.glob(os.path.join(folder_path, '*.csv'))

# Define error rate data frame
correct_rate_columns = ['participant_id', 
                        'correct_rate_bar_geo', 
                        'correct_rate_bar_icon', 
                        'correct_rate_bar_color', 
                        'correct_rate_pie_geo', 
                        'correct_rate_pie_icon', 
                        'correct_rate_pie_color']
df_correct_rate = pd.DataFrame(columns=correct_rate_columns)


# Define error rate data frame
response_time_columns = ['participant_id', 
                        'response_time_bar_geo', 
                        'response_time_bar_icon', 
                        'response_time_bar_color', 
                        'response_time_pie_geo', 
                        'response_time_pie_icon', 
                        'response_time_pie_color']
df_response_time = pd.DataFrame(columns=response_time_columns)

In [91]:
def get_correct_rate(df_file, texture_condition):
    """
    This function is to calculate the correct rate in a condition per participant.
    df_file: one file in individual _texture, which records all trials of a participant.
    """
    df_file_filtered = df_file[(df_file['trial_type'] == 'real') &
             (df_file['texture_condition'] == texture_condition)]
    # find out the real trials(not training trails), and the texture condition we want
    total_rows = len(df_file_filtered)
    correct_rows = len(df_file_filtered[df_file_filtered['answer_accuracy'] == 1])
    correct_rate = (correct_rows / total_rows) * 100
    
    return correct_rate

In [92]:
def get_overall_correct_rate(df_file):
    """
    This function is to calculate the overall correct rate per participant.
    df_file: one file in individual _texture, which records all trials of a participant.
    """
    df_file_filtered = df_file[(df_file['trial_type'] == 'real')]
    # find out the real trials(not training trails), and the texture condition we want
    total_rows = len(df_file_filtered)
    correct_rows = len(df_file_filtered[df_file_filtered['answer_accuracy'] == 1])
    correct_rate = (correct_rows / total_rows) * 100
    
    return correct_rate

### Response time

In [93]:
def get_response_time(df_file, texture_condition):
    """
    This function is to calculate the response time per participant in a condition. 
    We average all trials each participant did.
    df_file: one file in individual _texture, which records all trials of a participant.
    """
    df_file_filtered = df_file[(df_file['trial_type'] == 'real') &
             (df_file['texture_condition'] == texture_condition)& # find out the real trials(not training trails), and the texture condition we want
             (df_file['answer_accuracy'] == 1)] # We only count correct trials
    avg_response_time = df_file_filtered['elapsed_time'].mean()
    
    return avg_response_time

In [94]:
# def get_response_time(df_file, texture_condition):
#     """
#     This function is to calculate the response time per participant in a condition. We average all trials each participant did.
#     df_file: one file in individual _texture, which records all trials of a participant.
#     """
#     df_file_filtered = df_file[(df_file['trial_type'] == 'real') &
#              (df_file['texture_condition'] == texture_condition)& # find out the real trials(not training trails), and the texture condition we want
#              (df_file['answer_accuracy'] != -1)] # We exclude the time-out trials
#     avg_response_time = df_file_filtered['elapsed_time'].mean()
    
#     return avg_response_time

In [95]:
# def get_response_time(df_file, texture_condition):
#     """
#     This function is to calculate the response time per participant in a condition. We average all trials each participant did.
#     df_file: one file in individual _texture, which records all trials of a participant.
#     """
#     df_file_filtered = df_file[(df_file['trial_type'] == 'real') &
#              (df_file['texture_condition'] == texture_condition)] # find out the real trials(not training trails), and the texture condition we want

#     avg_response_time = df_file_filtered['elapsed_time'].mean()
    
#     return avg_response_time

In [ ]:
# Initialize an empty list to store valid individual DataFrames
dataframes = []
high_accuracy_participant = 0
high_accuracy_participants = []

# Read and store each CSV file as a DataFrame, and check if the filename matches a participant_id
for file in csv_files:
    # Get the filename without extension, which should be the participant's id
    file_name = os.path.splitext(os.path.basename(file))[0]

    
    # Check if the filename matches any value in the 'participant_id' column
    if file_name in df['participant_id'].values:
        # Read the CSV file and store it as a DataFrame
        df_file = pd.read_csv(file)
        
        # Add this participants' answer to df_correct_rate
        correct_rate_new_row = pd.Series({"participant_id": file_name}, name = "participant_id")
        df_correct_rate = pd.concat([df_correct_rate, correct_rate_new_row])
        
        correct_rate = get_correct_rate(df_file, 'geo')
        df_correct_rate.loc[df_correct_rate['participant_id'] == file_name, f"correct_rate_{df_file['condition'].iloc[0]}_geo"] = correct_rate
        
        correct_rate = get_correct_rate(df_file, 'icon')
        df_correct_rate.loc[df_correct_rate['participant_id'] == file_name, f"correct_rate_{df_file['condition'].iloc[0]}_icon"] = correct_rate
        
        correct_rate = get_correct_rate(df_file, 'color')
        df_correct_rate.loc[df_correct_rate['participant_id'] == file_name, f"correct_rate_{df_file['condition'].iloc[0]}_color"] = correct_rate
        
        # Add this participants' answer to df_response_time
        # Check if this participant's overall correct rate > 90%
        attention_check = True in ((df['participant_id'] == file_name) & (df['attention_check'] == 'Broccoli')).unique()
        distraction = True in ((df['participant_id'] == file_name) & (df['distraction'] == 'attention')).unique()
        commitmit = True in ((df['participant_id'] == file_name) & (df['commitment_ver'] == 'all')).unique()

        overall_correct_rate = get_overall_correct_rate(df_file)
        if (overall_correct_rate >= 90 and attention_check and distraction and commitmit):
            high_accuracy_participant = high_accuracy_participant + 1
            print(f"{file_name}.csv has {overall_correct_rate}% correct rate")
            high_accuracy_participants.append(file_name)
            response_time_new_row = pd.Series({"participant_id": file_name}, name = "participant_id")
            df_response_time = pd.concat([df_response_time, response_time_new_row])

            response_time = get_response_time(df_file, 'geo')
            df_response_time.loc[df_response_time['participant_id'] == file_name, f"response_time_{df_file['condition'].iloc[0]}_geo"] = response_time

            response_time = get_response_time(df_file, 'icon')
            df_response_time.loc[df_response_time['participant_id'] == file_name, f"response_time_{df_file['condition'].iloc[0]}_icon"] = response_time

            response_time = get_response_time(df_file, 'color')
            df_response_time.loc[df_response_time['participant_id'] == file_name, f"response_time_{df_file['condition'].iloc[0]}_color"] = response_time

        # Check if the number of rows is equal to 78
        dataframes.append(df_file)
        if df_file.shape[0] != 78:
            # Print the filename if the number of rows is not equal to 78 (Please note here the lost trials can be real trials or training trials)
            print(f"{file_name}.csv has {df_file.shape[0]} rows instead of 78")


# Concatenate all individual DataFrames into a single DataFrame
combined_df = pd.concat(dataframes, ignore_index=True)

# Check the combined DataFrame
# print(combined_df)

print(df_response_time)

combined_df.to_csv('generated_csv/combined.csv', index=False)

df_correct_rate.to_csv('generated_csv/correct_rate.csv', index=False)
df_correct_rate.to_csv('CI-analysis/exp-data/correct_rate.csv', index=False)
df_response_time.to_csv('generated_csv/response_time.csv', index=False)
df_response_time.to_csv('CI-analysis/exp-data/response_time.csv', index=False)

agosyvi.csv has 95.0% correct rate
aofvwlu.csv has 95.0% correct rate
bdotqhk.csv has 98.33333333333333% correct rate
fkzvuch.csv has 90.0% correct rate
hqcjtae.csv has 95.0% correct rate
kintsyd.csv has 98.33333333333333% correct rate
lnbkhse.csv has 100.0% correct rate
ovlndtq.csv has 93.33333333333333% correct rate
txjhcfs.csv has 91.66666666666666% correct rate
ueapycj.csv has 98.27586206896551% correct rate
ueapycj.csv has 76 rows instead of 78
ueiwhyc.csv has 91.66666666666666% correct rate
vjysdme.csv has 93.33333333333333% correct rate
xkyejci.csv has 95.0% correct rate
ybwelfj.csv has 93.33333333333333% correct rate
yvnfhsc.csv has 95.0% correct rate
zevwins.csv has 98.33333333333333% correct rate
               participant_id response_time_bar_geo response_time_bar_icon  \
participant_id        agosyvi           2514.055556            3048.473684   
participant_id        aofvwlu               3050.35            3507.529412   
participant_id        bdotqhk                1846.

In [97]:
high_accuracy_participant

16

In [98]:
combined_df["trial_type"].value_counts() # combined all trials from all validated participants

trial_type
real        1918
training     576
Name: count, dtype: int64

In [99]:
len(high_accuracy_participants)

16

## Distribution of missing trials

This section of scripts aims to determine how the missing trials are distributed among the high-accuracy participants.

In [100]:
# Read and store each CSV file as a DataFrame, and check if the filename matches a participant_id
for file in csv_files:
    # Get the filename without extension, which should be the participant's id
    file_name = os.path.splitext(os.path.basename(file))[0]

    
    # Check if the filename matches any value in the high_accuracy_participants
    if file_name in df['participant_id'].values:
        # Read the CSV file and store it as a DataFrame
        df_file = pd.read_csv(file)
        
        # Check number of rows of "real" experiment df_file have
        num_real = len(df_file[df_file['trial_type'] == 'real'])
        
        if (num_real != 60):
            if file_name in high_accuracy_participants:
                print(f"{file_name}.csv has {60 - num_real} rows missing [high accuracy participant]")
            else:
                print(f"{file_name}.csv has {60 - num_real} rows missing")

ueapycj.csv has 2 rows missing [high accuracy participant]


## Data analysis

### Aesthetic

In [101]:
# aesthetics
def writeBeauvisScore(df, condition):
    """
    calculate average score of 5 items in the BeauVis scale per participants (row), 
    and write it into a column in df (beauvis_{condition})
    """
    df[f"beauvis_{condition}"] = ""
    for index, row in df.iterrows():
        beauvis_score = np.mean(row[[f"beauvis0_{condition}", 
                                     f"beauvis1_{condition}",
                                     f"beauvis2_{condition}",
                                     f"beauvis3_{condition}",
                                     f"beauvis4_{condition}"
                                    ]])
        # update the value of df (the original dataframe)
        df.loc[index, f"beauvis_{condition}"] = beauvis_score

In [102]:
writeBeauvisScore(df, 'bar_geo')
writeBeauvisScore(df, 'bar_icon')
writeBeauvisScore(df, 'bar_color')
writeBeauvisScore(df, 'pie_geo')
writeBeauvisScore(df, 'pie_icon')
writeBeauvisScore(df, 'pie_color')

In [103]:
df_beauvis = df.loc[:, ['participant_id',
                        'beauvis_bar_geo',
                        'beauvis_bar_icon',
                        'beauvis_bar_color',
                        'beauvis_pie_geo',
                        'beauvis_pie_icon',
                        'beauvis_pie_color',]]
df_beauvis = df_beauvis[df_beauvis['participant_id'].isin(high_accuracy_participants)]
df_beauvis.to_csv('generated_csv/beauvis.csv', index=False)
df_beauvis.to_csv('CI-analysis/exp-data/beauvis.csv', index=False)

### Readability

In [104]:
# aesthetics
def writePrevisScore(df, condition):
    """
    calculate average score of 5 items in the BeauVis scale per participants (row), 
    and write it into a column in df (beauvis_{condition})
    """
    df[f"beauvis_{condition}"] = ""
    for index, row in df.iterrows():
        previs_score = np.mean(row[[f"previs0_{condition}", 
                                     f"previs1_{condition}",
                                     f"previs2_{condition}",
                                     f"previs3_{condition}",
                                     f"previs4_{condition}",
                                     f"previs5_{condition}", 
                                     f"previs6_{condition}",
                                     f"previs7_{condition}",
                                    #  f"previs8_{condition}",
                                    #  f"previs9_{condition}",
                                     f"previs10_{condition}"
                                    ]])
        # update the value of df (the original dataframe)
        df.loc[index, f"previs_{condition}"] = previs_score

In [105]:
writePrevisScore(df, 'bar_geo')
writePrevisScore(df, 'bar_icon')
writePrevisScore(df, 'bar_color')
writePrevisScore(df, 'pie_geo')
writePrevisScore(df, 'pie_icon')
writePrevisScore(df, 'pie_color')

In [106]:
df_readable = df.loc[:, ['participant_id',
                        'previs_bar_geo',
                        'previs_bar_icon',
                        'previs_bar_color',
                        'previs_pie_geo',
                        'previs_pie_icon',
                        'previs_pie_color',]]
df_readable = df_readable[df_readable['participant_id'].isin(high_accuracy_participants)]

df_readable.to_csv('generated_csv/readable.csv', index=False)
df_readable.to_csv('CI-analysis/exp-data/readable.csv', index=False)

### Time Out
analysis the distribution of time out trials

In [107]:
# Generate the subset DataFrame of time out real trials
df_timeout_pre = combined_df[(combined_df['trial_type'] == 'real') & (combined_df['answer_accuracy'] == -1)]
df_timeout_pre.to_csv('generated_csv/timeout.csv', index=False)
df_timeout_pre = df_timeout_pre.reset_index(drop=True)

In [108]:
df_timeout = pd.read_csv('generated_csv/timeout.csv')

In [109]:
# Fix df_timeout, because for some trials, the key_pressed was not logged.
# function to apply on each row
def shift_values(row):
    if row['key_pressed'] != 'none':
        row.loc['key_pressed':] = row.loc['key_pressed':].shift(1)
        row['key_pressed'] = 'none'
    return row

# apply the function to each row
df_timeout = df_timeout.apply(shift_values, axis=1)
df_timeout.to_csv('generated_csv/timeout_fixed.csv', index=False)

In [110]:
df_timeout['condition'].value_counts()

condition
bar    21
pie    20
Name: count, dtype: int64

In [111]:
df_timeout['texture_condition'].value_counts()

texture_condition
geo      17
icon     17
color     7
Name: count, dtype: int64

In [112]:
print("number of time-out trials in...")
print("each chart type:")
print(df_timeout['condition'].value_counts())
print()

print("each texture type:")
print(df_timeout['texture_condition'].value_counts())
print()


print("each condition")
filtered_df = df_timeout[(df_timeout['condition'] == 'bar') & (df_timeout['texture_condition'] == 'geo')]
num_rows = filtered_df.shape[0]
print("geometric bar:" + str(num_rows))
                                                               
filtered_df = df_timeout[(df_timeout['condition'] == 'bar') & (df_timeout['texture_condition'] == 'icon')]
num_rows = filtered_df.shape[0]
print("iconic bar:" + str(num_rows))
                                                               
filtered_df = df_timeout[(df_timeout['condition'] == 'bar') & (df_timeout['texture_condition'] == 'color')]
num_rows = filtered_df.shape[0]
print("gray bar:" + str(num_rows))

print()
filtered_df = df_timeout[(df_timeout['condition'] == 'pie') & (df_timeout['texture_condition'] == 'geo')]
num_rows = filtered_df.shape[0]
print("geometric pie:" + str(num_rows))
                                                               
filtered_df = df_timeout[(df_timeout['condition'] == 'pie') & (df_timeout['texture_condition'] == 'icon')]
num_rows = filtered_df.shape[0]
print("iconic pie:" + str(num_rows))
                                                               
filtered_df = df_timeout[(df_timeout['condition'] == 'pie') & (df_timeout['texture_condition'] == 'color')]
num_rows = filtered_df.shape[0]
print("gray pie:" + str(num_rows))

number of time-out trials in...
each chart type:
condition
bar    21
pie    20
Name: count, dtype: int64

each texture type:
texture_condition
geo      17
icon     17
color     7
Name: count, dtype: int64

each condition
geometric bar:9
iconic bar:7
gray bar:5

geometric pie:8
iconic pie:10
gray pie:2
